# 📦 SQLite Veritabanı Destekli Stok Takip Uygulaması

Bu proje, yerel bir **SQLite** veritabanı kullanarak ürünlerin stok kayıtlarını tutan, ekleme, silme, güncelleme (düzeltme), arama ve temizleme işlemlerini gerçekleştiren modern bir masaüstü **Stok Takip** uygulamasıdır.

Bu Jupyter Notebook dosyasında, uygulamanın adım adım nasıl çalıştığını, veritabanı işlemlerinin nasıl yapıldığını ve GUI (Grafik Arayüz) elemanlarının nasıl oluşturulduğunu öğrenebilirsiniz.

## 1. Adım: Gerekli Kütüphanelerin İçe Aktarılması

Uygulamamızda arayüz oluşturmak için `tkinter` ve `ttk`, verileri kalıcı olarak saklamak için yerleşik `sqlite3` ve dosya yolu yönetimleri için `os` kütüphanelerini kullanacağız.

In [ ]:
import os
import tkinter as tk
from tkinter import ttk
import sqlite3

## 2. Adım: Veritabanı ve Tablo Oluşturma Mantığı

Stok verilerini kaydetmek için programın çalıştığı klasörde `stok_takip.db` adında bir veritabanı dosyası oluşturup bağlantı açacağız. Ardından stok bilgilerini saklayacak tabloyu tanımlayacağız.

In [ ]:
# Veritabanı bağlantısı ve tablo oluşturma şeması:
db_path = "stok_takip_test.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 'stok' tablosunu oluşturma sorgusu
cursor.execute("""
     CREATE TABLE IF NOT EXISTS stok(
                                     id TEXT PRIMARY KEY,
                                     urun_adi TEXT,
                                     adet INTEGER,
                                     birim_fiyat REAL,
                                     toplam_deger REAL
                                     )        
""")
conn.commit()
print("Veritabanı ve stok tablosu başarıyla oluşturuldu!")

# Test bağlantısını kapatalım
conn.close()
if os.path.exists(db_path):
    os.remove(db_path) # Test dosyasını temizleyelim

## 3. Adım: CRUD (Ekleme, Listeleme, Güncelleme, Silme) İşlemleri

Uygulamanın veri tabanı mantığında CRUD fonksiyonları şu şekilde çalışır:

1. **Ekleme (`INSERT`)**: Formdaki verileri `stok` tablosuna ekler.
2. **Listeleme (`SELECT`)**: Tüm verileri tablodan çekip arayüzdeki `Treeview` tablosuna yansıtır.
3. **Güncelleme (`UPDATE`)**: Seçili ürünün ID'sine göre yeni bilgileri kaydeder.
4. **Silme (`DELETE`)**: Seçili ürünün kaydını veritabanından tamamen kaldırır.

## 4. Adım: Arayüz Elemanlarının Çizilmesi ve Mainloop

Aşağıdaki blokta uygulamanın arayüzünü başlatan tam kod yer almaktadır. Kodu çalıştırarak arayüzü görebilir, test kayıtları ekleyebilirsiniz.

**NOT:** Arayüz penceresi açıldığında çalışmaya devam edecektir. Kapatmak için arayüz penceresinin sağ üstündeki çarpı butonunu kullanabilirsiniz.

In [ ]:
class StokTakipUygulamasi:
    def __init__(self, root):
        self.root = root
        self.root.title("Stok Takip Uygulaması - Jupyter Test")
        self.root.resizable(False, False)

        # Veri Tabanını oluştur/bağlan
        db_dir = os.path.dirname(os.path.abspath("__file__"))
        db_path = os.path.join(db_dir, "stok_takip.db")
        self.conn = sqlite3.connect(db_path)
        self.cursor = self.conn.cursor()
        self.cursor.execute("""
             CREATE TABLE IF NOT EXISTS stok(
                                             id TEXT PRIMARY KEY,
                                             urun_adi TEXT,
                                             adet INTEGER,
                                             birim_fiyat REAL,
                                             toplam_deger REAL
                                             )        
        """)
        self.conn.commit()

        # Giriş Alanları
        self.id_label = tk.Label(root, text="Ürün ID: ")
        self.id_label.grid(row=0, column=0, padx=5, pady=5, sticky="w")
        self.id_entry = tk.Entry(root)
        self.id_entry.grid(row=0, column=1, padx=5, pady=5)
        
        self.urun_adi_label = tk.Label(root, text="Ürün Adı: ")
        self.urun_adi_label.grid(row=1, column=0, padx=5, pady=5, sticky="w")
        self.urun_adi_entry = tk.Entry(root)
        self.urun_adi_entry.grid(row=1, column=1, padx=5, pady=5)
        
        self.adet_label = tk.Label(root, text="Adet: ")
        self.adet_label.grid(row=2, column=0, padx=5, pady=5, sticky="w")
        self.adet_entry = tk.Entry(root)
        self.adet_entry.grid(row=2, column=1, padx=5, pady=5)

        self.birim_fiyati_label = tk.Label(root, text="Birim Fiyatı: ")
        self.birim_fiyati_label.grid(row=3, column=0, padx=5, pady=5, sticky="w")
        self.birim_fiyati_entry = tk.Entry(root)
        self.birim_fiyati_entry.grid(row=3, column=1, padx=5, pady=5)

        # İşlem Butonları
        self.ekle_buton = tk.Button(root, text="Ekle", command=self.ekle, width=10)
        self.ekle_buton.grid(row=4, column=0, padx=5, pady=10)
        
        self.duzelt_buton = tk.Button(root, text="Düzelt", command=self.duzelt, width=10)
        self.duzelt_buton.grid(row=4, column=1, padx=5, pady=10)

        self.sil_buton = tk.Button(root, text="Sil", command=self.sil, width=10)
        self.sil_buton.grid(row=4, column=2, padx=5, pady=10)
        
        self.temizle_buton = tk.Button(root, text="Temizle", command=self.girisleri_temizle, width=10)
        self.temizle_buton.grid(row=4, column=3, padx=5, pady=10)

        # Arama Çubuğu
        self.arama_label = tk.Label(root, text="Ara: ")
        self.arama_label.grid(row=5, column=0, padx=5, pady=5, sticky="w")
        self.arama_entry = tk.Entry(root)
        self.arama_entry.grid(row=5, column=1, padx=5, pady=5, columnspan=2, sticky="ew")
        self.arama_entry.bind("<KeyRelease>", self.arama)

        # Tablo Yapısı
        self.tablo = ttk.Treeview(root, columns=("ID", "Ürün Adı", "Adet", "Birim Fiyatı", "Toplam Değer"), show="headings")
        self.tablo.heading("ID", text="ID")
        self.tablo.heading("Ürün Adı", text="Ürün Adı")
        self.tablo.heading("Adet", text="Adet")
        self.tablo.heading("Birim Fiyatı", text="Birim Fiyatı")
        self.tablo.heading("Toplam Değer", text="Toplam Değer")
        
        self.tablo.column("ID", width=80, anchor="center")
        self.tablo.column("Ürün Adı", width=150, anchor="w")
        self.tablo.column("Adet", width=80, anchor="center")
        self.tablo.column("Birim Fiyatı", width=100, anchor="e")
        self.tablo.column("Toplam Değer", width=120, anchor="e")
        self.tablo.grid(row=6, column=0, columnspan=4, padx=10, pady=10)

        self.tablo.bind("<ButtonRelease-1>", self.satir_sec)
        self.verileri_yukle()

    def ekle(self):
        try:
            id = self.id_entry.get()
            urun_adi = self.urun_adi_entry.get()
            adet = int(self.adet_entry.get())
            birim_fiyat = float(self.birim_fiyati_entry.get())
            toplam_deger = adet * birim_fiyat

            self.cursor.execute("INSERT INTO stok VALUES(?,?,?,?,?)", (id, urun_adi, adet, birim_fiyat, toplam_deger))
            self.conn.commit()
            self.tablo.insert("", "end", values=(id, urun_adi, adet, birim_fiyat, toplam_deger))
            self.girisleri_temizle()
        except ValueError:
            pass

    def girisleri_temizle(self):
        self.id_entry.delete(0, tk.END)
        self.urun_adi_entry.delete(0, tk.END)
        self.adet_entry.delete(0, tk.END)
        self.birim_fiyati_entry.delete(0, tk.END)

    def arama(self, event):
        arama_metni = self.arama_entry.get().lower()
        for item in self.tablo.get_children():
            values = self.tablo.item(item, "values")
            if arama_metni in values[0].lower() or arama_metni in values[1].lower() or arama_metni in values[2].lower() or arama_metni in values[3].lower():
                self.tablo.selection_set(item)
                self.tablo.see(item)
            else:
                self.tablo.selection_remove(item)

    def satir_sec(self, event):
        secili = self.tablo.selection()
        if secili:
            item = self.tablo.item(secili)
            values = item["values"]
            self.girisleri_temizle()
            self.id_entry.insert(0, values[0])
            self.urun_adi_entry.insert(0, values[1])
            self.adet_entry.insert(0, values[2])
            self.birim_fiyati_entry.insert(0, values[3])

    def duzelt(self):
        secili = self.tablo.selection()
        if secili:
            try:
                id = self.id_entry.get()
                urun_adi = self.urun_adi_entry.get()
                adet = int(self.adet_entry.get())
                birim_fiyat = float(self.birim_fiyati_entry.get())
                toplam_deger = adet * birim_fiyat

                self.cursor.execute("UPDATE stok SET urun_adi = ?, adet = ?, birim_fiyat = ?, toplam_deger = ? WHERE id = ?", (urun_adi, adet, birim_fiyat, toplam_deger, id))
                self.conn.commit()
                self.tablo.item(secili, values=(id, urun_adi, adet, birim_fiyat, toplam_deger))
                self.girisleri_temizle()
            except ValueError:
                pass

    def sil(self):
        secili = self.tablo.selection()
        if secili:
            id = self.tablo.item(secili)['values'][0]
            self.cursor.execute("DELETE FROM stok WHERE id = ?", (id,))
            self.conn.commit()
            self.tablo.delete(secili)
            self.girisleri_temizle()

    def verileri_yukle(self):
        for row in self.cursor.execute("SELECT * FROM stok"):
            self.tablo.insert("", "end", values=row)

# Test uygulamasını çalıştıralım
root = tk.Tk()
app = StokTakipUygulamasi(root)
root.mainloop()

---

# 🔄 Programın Çalışma Akışı ve Yapısı

Stok Takip uygulaması çalıştırıldığında işletim sistemi ve veritabanı ile sürekli etkileşim halinde olur:

1. **Veritabanı Hazırlığı**: Uygulama klasöründe `stok_takip.db` SQLite veritabanı dosyası aranır, yoksa otomatik oluşturulup tablo şeması kurulur.
2. **Arayüz Kurulumu (GUI)**: Tkinter penceresi açılarak giriş etiketleri, CRUD butonları, arama filteresinden ve verileri gösteren tablo (`Treeview`) çizilir.
3. **Veri Yükleme**: Veritabanında daha önceden kayıtlı olan tüm ürün verileri `SELECT *` sorgusu ile çekilerek `Treeview` tablosuna doldurulur.
4. **Kullanıcı Etkileşimi (CRUD & Arama)**: 
   - **Ekleme**: Girilen verileri doğrular, SQLite `INSERT` ile kaydeder, Treeview tablosunun sonuna ekler.
   - **Düzeltme**: Tablodan tıklanarak seçilen satırdaki ürünü `UPDATE` sorgusu ile günceller.
   - **Silme**: Seçilen ürünü `DELETE` sorgusu ile veritabanından ve arayüz tablosundan siler.
   - **Arama**: Arama çubuğuna yazılan metni klavyenin tuş bırakma olayında (`<KeyRelease>`) algılar ve tablodaki eşleşen satırları seçili yapar.